# Aula 04 · Figuras do Lab 1

Gera os dados das figuras dos slides do Lab 1: amostras das 10 classes do Fashion-MNIST, as curvas da configuração do checkpoint (MLP 784 → 256 → 256 → 10, AdamW, η = 0,001, batch 128, 15 épocas, split 55.000 / 5.000) e a matriz de confusão na validação.

Roda em CPU em menos de um minuto. As figuras SVG são montadas por `tools/aula04_figuras_svg.py dados/lab01_dados.json <saida>`.

In [ ]:
import json, os, time, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torchvision import datasets
os.makedirs('dados', exist_ok=True)
ds = datasets.FashionMNIST('data', train=True, download=True)
X, y = ds.data.reshape(-1, 784).numpy() / 255., ds.targets.numpy()
CLASSES = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
torch.set_num_threads(4); torch.manual_seed(42)
res = {'classes': CLASSES, 'mean': float(X.mean()), 'std': float(X.std())}
r = np.random.default_rng(0)
res['samples'] = [[X[i].reshape(28, 28).round(3).tolist() for i in r.choice(np.where(y == c)[0], 3, replace=False)] for c in range(10)]
Xn = (X - 0.2860) / 0.3530
p = torch.randperm(60000, generator=torch.Generator().manual_seed(42))
tr, va = p[:55000], p[55000:]
Xt, yt = torch.tensor(Xn[tr.numpy()], dtype=torch.float32), torch.tensor(y[tr.numpy()])
Xv, yv = torch.tensor(Xn[va.numpy()], dtype=torch.float32), torch.tensor(y[va.numpy()])
net = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 10))
res['params'] = sum(q.numel() for q in net.parameters())
opt = torch.optim.AdamW(net.parameters(), 1e-3); hist = []; t0 = time.time()
for ep in range(15):
    net.train(); q = torch.randperm(55000); tl = tc = 0
    for k in range(0, 55000, 128):
        b = q[k:k + 128]; out = net(Xt[b]); loss = F.cross_entropy(out, yt[b])
        opt.zero_grad(); loss.backward(); opt.step()
        tl += loss.item() * len(b); tc += (out.argmax(1) == yt[b]).sum().item()
    net.eval()
    with torch.no_grad():
        o = net(Xv); vl = F.cross_entropy(o, yv).item(); vacc = (o.argmax(1) == yv).float().mean().item()
    hist.append((tl / 55000, vl, tc / 55000, vacc)); print(ep + 1, np.round(hist[-1], 4))
res['hist'] = hist; res['tempo_s'] = time.time() - t0
with torch.no_grad():
    pred = net(Xv).argmax(1).numpy()
cm = np.zeros((10, 10), int)
for a, b in zip(yv.numpy(), pred): cm[a, b] += 1
res['confusion'] = cm.tolist()
json.dump(res, open('dados/lab01_dados.json', 'w'))
print('tempo', round(res['tempo_s']), 'params', res['params'])